# 04 · Striker scouting profiles

**Question:** Which strikers in this sample generate non-penalty chances and contribute to attacking play, warranting further scouting?

Run after notebook 03. We will expand the event sample, audit participation time, construct striker profiles, and apply transparent eligibility rules. This is a retrospective tournament analysis, not a validated recruitment prediction or market-value model.

The full tournament is still a short, selected international sample. Club performance, age, wages, fees, injuries and transfer availability are absent. No single composite ranking is produced.


In [ ]:
import sys
import csv
import json
from collections import Counter, defaultdict
from pathlib import Path

assert sys.version_info >= (3, 11), "Use a Python 3.11+ kernel."
PROJECT_OVERRIDE = None  # Optional: Path("/path/to/soccer-recruitment")
if PROJECT_OVERRIDE is not None:
    candidates = [Path(PROJECT_OVERRIDE).expanduser()]
else:
    candidates = []
    for folder in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([folder, folder / "soccer-recruitment"])
ROOT = next((p.resolve() for p in candidates if (p / "src/soccer/cli.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory, or set PROJECT_OVERRIDE to your complete checkout.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
for folder in ("reports", "data/processed"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("Project:", ROOT.name)

paths = sorted((ROOT / "data/raw/statsbomb/events").glob("*.json"))
if not paths:
    raise FileNotFoundError("Run notebook 01 with acquisition enabled first.")
print("Downloaded event matches:", len(paths))


## 1. Expand beyond the starter matches

Set `EXPAND_COHORT=True` to download all 64 2022 men's World Cup matches from the pinned source revision, including lineups. This can use hundreds of MB and take several minutes. Existing cached files are verified and reused. The acquisition function also checks the existing Metrica sample; we do not join it to named strikers.

Set the flag to False to explore only your existing matches. Acquisition is additive; inspect the actual cohort printed below. If you have downloaded other competitions into this project, this notebook explicitly limits analysis to matches in the 2022 World Cup catalog.


In [ ]:
import sys
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from soccer.acquire import fetch

EXPAND_COHORT = True
MIN_MINUTES = 300  # Illustrative eligibility threshold; not a guarantee of reliability.
MIN_STRIKER_EVENT_SHARE = 0.5
if EXPAND_COHORT:
    fetch(ROOT, competition=43, season=106, match_count=64, game=1)
else:
    print("Using existing event files only.")


In [ ]:
catalog = json.loads((ROOT / "data/raw/statsbomb/matches/43_106.json").read_text(encoding="utf-8"))
match_ids = {str(m["match_id"]) for m in catalog}
events_by_match = {p.stem: json.loads(p.read_text(encoding="utf-8"))
                   for p in sorted((ROOT / "data/raw/statsbomb/events").glob("*.json"))
                   if p.stem in match_ids}
print(f"Analyzing {len(events_by_match)} of {len(match_ids)} catalog matches.")
print("Events:", sum(map(len, events_by_match.values())))
if not events_by_match:
    raise ValueError("No eligible match files. Enable cohort expansion.")
if EXPAND_COHORT and len(events_by_match) != len(match_ids):
    raise ValueError("Incomplete tournament download; inspect missing files.")


## 2. Reuse the participation-time method

The next cell contains the same calculation as notebook 03: actual elapsed periods including stoppage time, excluding halftime and shootouts, with starters/substitutions/dismissals audited against lineup rosters. Temporary injury absences remain included. It is included here so you do not need to run another notebook in the same kernel.

If an unusual transition fails the audit, review it before proceeding; do not suppress the error just to obtain results. Extra time counts as participation. Shootout shots and goals are excluded from all features.


In [ ]:
def seconds(timestamp):
    h, m, s = timestamp.split(":")
    return int(h)*3600 + int(m)*60 + float(s)

def playing_time(events, roster_ids):
    played_periods = sorted({e["period"] for e in events if 1 <= e["period"] <= 4})
    if played_periods != list(range(1, max(played_periods)+1)):
        raise ValueError("Missing played period")
    durations = {}
    offsets = {}
    for period in played_periods:
        ends = [seconds(e["timestamp"]) for e in events if e["period"] == period and e["type"]["name"] == "Half End"]
        if not ends:
            raise ValueError(f"Period {period} has no Half End")
        offsets[period] = sum(durations.values())
        durations[period] = max(ends)
    total = sum(durations.values())
    active, rows, intervals = {}, {}, []
    removed = set()
    temporary_off = Counter()
    red_cards = Counter()
    permanent_off = Counter()
    bench_discipline_events = 0

    def ensure(team, player):
        key = (team["id"], player["id"])
        if key not in roster_ids:
            raise ValueError(f"Player absent from lineup roster: {key}")
        rows.setdefault(key, dict(team_id=team["id"], team=team["name"], player_id=player["id"], player=player["name"], minutes=0.0))
        return key

    def close(key, t, reason):
        if key not in active:
            raise ValueError(f"{reason}: player not active: {key}")
        start = active.pop(key)
        if t < start:
            raise ValueError("Negative participation interval")
        rows[key]["minutes"] += (t-start)/60
        intervals.append((key, start, t, reason))
        removed.add(key)

    for e in events:
        if e["type"]["name"] == "Starting XI":
            for slot in e["tactics"]["lineup"]:
                key = ensure(e["team"], slot["player"])
                if key in active:
                    raise ValueError("Duplicate starter")
                active[key] = 0.0
    teams = {key[0] for key in active}
    if len(teams) != 2 or any(sum(k[0] == team for k in active) != 11 for team in teams):
        raise ValueError("Expected exactly two Starting XIs of eleven players")
    ordered = sorted((e for e in events if e["period"] in durations), key=lambda e: (e["period"], seconds(e["timestamp"]), e.get("index", 0)))
    for e in ordered:
        t = offsets[e["period"]] + seconds(e["timestamp"])
        if seconds(e["timestamp"]) > durations[e["period"]] + 0.001:
            raise ValueError("Event occurs after period end")
        kind = e["type"]["name"]
        if kind == "Substitution":
            outgoing = ensure(e["team"], e["player"])
            # A prior permanent-departure event may already have ended this player's interval.
            if outgoing in active:
                close(outgoing, t, "substitution")
            elif outgoing not in removed:
                raise ValueError("Outgoing substitute never active")
            incoming = ensure(e["team"], e["substitution"]["replacement"])
            if incoming in active or incoming in removed:
                raise ValueError("Unsupported substitute re-entry")
            active[incoming] = t
        if kind == "Player Off":
            key = ensure(e["team"], e["player"])
            if e.get("player_off", {}).get("permanent", False):
                permanent_off[key[0]] += 1
                if key in active:
                    close(key, t, "permanent departure")
            else:
                temporary_off[key] += 1
        if kind == "Bad Behaviour" and e.get("position", {}).get("name") == "Substitute":
            bench_discipline_events += 1
            continue  # Bench discipline does not create participation time or an on-field dismissal.
        card = e.get("foul_committed", {}).get("card", {}).get("name") or e.get("bad_behaviour", {}).get("card", {}).get("name")
        if card in ("Red Card", "Second Yellow"):
            key = ensure(e["team"], e["player"])
            if key in active:
                close(key, t, card)
                red_cards[key[0]] += 1
            else:
                raise ValueError(f"Dismissal of inactive player needs manual review: {key}")
    for key in list(active):
        close(key, total, "match end")
    for key, row in rows.items():
        row["temporary_off_events"] = temporary_off[key]
        if not 0 <= row["minutes"] <= total/60 + 1e-8:
            raise ValueError("Invalid minutes")
    audit = dict(match_elapsed_minutes=total/60, period_minutes={str(p):d/60 for p,d in durations.items()},
                 temporary_off_events=sum(temporary_off.values()), dismissals=sum(red_cards.values()),
                 permanent_departures=sum(permanent_off.values()), participants=len(rows), bench_discipline_events=bench_discipline_events)
    for team in teams:
        actual = sum(r["minutes"] for k,r in rows.items() if k[0] == team)
        expected = 11*total/60
        if actual > expected + 1e-6:
            raise ValueError("Team minutes exceed eleven-player maximum")
        if not red_cards[team] and not permanent_off[team] and abs(actual-expected) > 1e-6:
            raise ValueError("Team participation time does not reconcile")
    return rows, audit


In [ ]:
minutes_by_match, audits = {}, {}
for match_id, events in events_by_match.items():
    roster = json.loads((ROOT / f"data/raw/statsbomb/lineups/{match_id}.json").read_text(encoding="utf-8"))
    roster_ids = {(team["team_id"], p["player_id"]) for team in roster for p in team["lineup"]}
    minutes_by_match[match_id], audits[match_id] = playing_time(events, roster_ids)
print("Audited matches:", len(audits))
print("Temporary departure events (included in participation time):", sum(a["temporary_off_events"] for a in audits.values()))


## 3. Define a striker cohort and attacking features

**Role proxy:** Center Forward, Left Center Forward, Right Center Forward, or Secondary Striker on at least half of a player's positioned events. This event-weighted share is not time spent in a role: on-ball involvement can bias it. Total minutes and features include all of the player's roles. Audit borderline cases manually; do not describe these as striker-only minutes.

**Features:** non-penalty shots, provider non-penalty xG (npxG), non-penalty goals, npxG per shot, penalty-area shot share, shot assists, passes and pressures. Shot assists count passes marked `shot_assist`, regardless of whether the resulting shot was a penalty. Penalty-area shots use StatsBomb coordinates x ≥ 102 and 18 ≤ y ≤ 62 on the 120×80 event grid. That is provider-coordinate geometry, not Metrica meters.

Non-penalty includes open-play and non-penalty set-piece shots. Goals minus npxG describes this sample, not sustainable finishing skill. Provider xG is accepted as supplied, not fitted here. Pressures count actions without measuring success.


In [ ]:
STRIKER_ROLES = {"Center Forward", "Left Center Forward", "Right Center Forward", "Secondary Striker"}
players = {}
excluded_bench_events = []
for match_id, events in events_by_match.items():
    for key, exposure in minutes_by_match[match_id].items():
        row = players.setdefault(key, dict(team_id=key[0], team=exposure["team"], player_id=key[1], player=exposure["player"],
            matches=0, minutes=0.0, positioned_events=0, striker_events=0, nonpen_shots=0, npxg=0.0,
            nonpen_goals=0, nonpen_box_shots=0, shot_assists=0, passes=0, pressures=0))
        row["minutes"] += exposure["minutes"]
        row["matches"] += 1
    for e in events:
        if e["period"] == 5 or "player" not in e:
            continue
        key = (e["team"]["id"], e["player"]["id"])
        if e["type"]["name"] == "Bad Behaviour" and e.get("position", {}).get("name") == "Substitute":
            excluded_bench_events.append(dict(match_id=match_id, event_id=e.get("id"), player_id=key[1], player=e["player"]["name"], reason="Discipline recorded for a substitute"))
            continue
        if key not in minutes_by_match[match_id]:
            raise ValueError(f"Unmatched playing action: {match_id}, {key}, {e['type']['name']}")
        row = players[key]
        role = e.get("position", {}).get("name")
        if role:
            row["positioned_events"] += 1
            row["striker_events"] += int(role in STRIKER_ROLES)
        kind = e["type"]["name"]
        if kind == "Shot" and e["shot"]["type"]["name"] != "Penalty":
            row["nonpen_shots"] += 1
            row["npxg"] += e["shot"].get("statsbomb_xg", 0.0)
            row["nonpen_goals"] += int(e["shot"]["outcome"]["name"] == "Goal")
            loc = e.get("location")
            row["nonpen_box_shots"] += int(bool(loc and loc[0] >= 102 and 18 <= loc[1] <= 62))
        elif kind == "Pass":
            row["passes"] += 1
            row["shot_assists"] += int(e["pass"].get("shot_assist", False))
        elif kind == "Pressure":
            row["pressures"] += 1

profiles = []
for key, row in sorted(players.items()):
    row["striker_event_share"] = row["striker_events"]/row["positioned_events"] if row["positioned_events"] else 0.0
    row["npxg_per_shot"] = row["npxg"]/row["nonpen_shots"] if row["nonpen_shots"] else None
    row["nonpen_box_shot_share"] = row["nonpen_box_shots"]/row["nonpen_shots"] if row["nonpen_shots"] else None
    row["nonpen_goals_minus_npxg"] = row["nonpen_goals"] - row["npxg"]
    for metric in ("nonpen_shots", "npxg", "nonpen_goals", "shot_assists", "passes", "pressures"):
        row[metric+"_per90"] = 90*row[metric]/row["minutes"] if row["minutes"] > 0 else None
    row["eligible"] = row["minutes"] >= MIN_MINUTES and row["striker_event_share"] >= MIN_STRIKER_EVENT_SHARE
    profiles.append(row)
strikers = [r for r in profiles if r["striker_event_share"] >= MIN_STRIKER_EVENT_SHARE]
eligible = [r for r in profiles if r["eligible"]]
print("Player-team profiles:", len(profiles))
print("Striker-role proxy matches:", len(strikers))
print(f"Eligible at {MIN_MINUTES} minutes:", len(eligible))
if not eligible:
    print("No eligible players yet. Expand the cohort; do not quietly lower the exposure threshold.")
for row in sorted(eligible, key=lambda r:(r["team"],r["player"])):
    print(f"{row['player'][:28]:28} | {row['minutes']:6.1f} min | shots/90 {row['nonpen_shots_per90']:4.2f} | npxG/90 {row['npxg_per90']:4.2f} | npxG/shot {row['npxg_per_shot'] if row['npxg_per_shot'] is not None else 'NA'}")

print("Excluded bench discipline events:", len(excluded_bench_events))


## 4. Verify and save

Reconcile non-penalty shot totals and provider xG against source events. Save all player profiles as well as the eligible striker subset, preserving eligibility flags and raw denominators. There may legitimately be no eligible rows in the starter sample.


In [ ]:
import math
shots = [e for events in events_by_match.values() for e in events
         if e["period"] != 5 and e["type"]["name"] == "Shot" and "player" in e and e["shot"]["type"]["name"] != "Penalty"]
assert sum(r["nonpen_shots"] for r in profiles) == len(shots)
assert math.isclose(sum(r["npxg"] for r in profiles), sum(e["shot"].get("statsbomb_xg",0.0) for e in shots))
assert all(0 <= r["nonpen_goals"] <= r["nonpen_shots"] for r in profiles)
assert all(0 <= r["striker_event_share"] <= 1 for r in profiles)
for filename, output_rows in [("striker_all_player_profiles.csv", profiles), ("striker_eligible_profiles.csv", eligible)]:
    path = ROOT / "reports" / filename
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(profiles[0]))
        writer.writeheader()
        writer.writerows(output_rows)
    print("Saved:", path)
manifest = dict(competition=43, season=106, analyzed_match_ids=sorted(events_by_match),
                minimum_minutes=MIN_MINUTES, minimum_striker_event_share=MIN_STRIKER_EVENT_SHARE,
                striker_roles=sorted(STRIKER_ROLES), minutes_convention="elapsed participation including stoppage and temporary absences; shootouts excluded",
                source_revisions=json.loads((ROOT / "sources.lock.json").read_text(encoding="utf-8"))["revisions"])
(ROOT / "reports/striker_analysis_manifest.json").write_text(json.dumps(manifest,indent=2)+"\n",encoding="utf-8")
print("Arithmetic checks passed.")

(ROOT / "reports/striker_excluded_bench_events.json").write_text(json.dumps(excluded_bench_events, indent=2)+"\n", encoding="utf-8")


## Read the profiles as scouting questions

- High shots/90 with low npxG/shot: is the player taking difficult shots, or does the team fail to create better chances?
- High npxG/shot with low shot volume: does the player find valuable positions but receive too few opportunities?
- Goals above npxG: could this be short-sample variation rather than repeatable finishing?
- High pressure counts: does the tactical system require more pressing, and are the actions effective?

All of these require context and video review. Participation and role filters do not remove team strength, opponent, match-state or tournament-selection effects. Check sensitivity to 180/300/450 minutes and role thresholds without selecting the setting that makes a preferred player look best.

**Next:** review the cohort and choose two or three striker profiles for contextual comparison. More seasons of club data would be needed for robust prospective recruitment evaluation. This World Cup analysis alone cannot establish undervaluation. Any predictive model must use pre-decision features, future targets, chronological evaluation, training-only preprocessing and appropriate player/match grouping; this notebook does not validate one.
